In [1]:
# 1. Install Rust
!curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y
import os
os.environ['PATH'] += ":/root/.cargo/bin"

# 2. Install Kani Verifier (This is what makes it "Verifiable")
!cargo install --locked kani-verifier
!cargo kani setup

info: downloading installer
warn: It looks like you have an existing rustup settings file at:
warn: /root/.rustup/settings.toml
warn: Rustup will install the default toolchain as specified in the settings file,
warn: instead of the one inferred from the default host triple.
info: profile set to default
info: default host triple is x86_64-unknown-linux-gnu
info: syncing channel updates for stable-x86_64-unknown-linux-gnu
info: latest update on 2026-03-05 for version 1.94.0 (4a4ef493e 2026-03-02)
info: downloading 6 components
        cargo downloading [               ]   10.48 MiB (0 B/s, ETA: 0s)
        cargo downloading [               ]   10.48 MiB (0 B/s, ETA: 0s)
        cargo downloading [               ]   10.48 MiB (0 B/s, ETA: 0s)
        cargo downloading [#              ]   10.48 MiB (2.88 MiB/s, ETA: 4s)
        cargo downloading [#              ]   10.48 MiB (2.88 MiB/s, ETA: 4s)
        cargo downloading [#              ]   10.48 MiB (2.88 MiB/s, ETA: 4s)
        cargo do

In [2]:
# Write the Rust code to a file
with open("aggregator.rs", "w") as f:
    f.write("""
#[kani::proof]
fn verify_update_bounds() {
    // We simulate a model update (gradient) from a node
    let update: f32 = kani::any();

    // Logic: If the update is within safe bounds, we accept it
    let result = safe_aggregate(update);

    // Formal Verification: Prove that the result is NEVER outside -1 to 1
    assert!(result >= -1.0 && result <= 1.0);
}

fn safe_aggregate(val: f32) -> f32 {
    if val > 1.0 {
        return 1.0;
    } else if val < -1.0 {
        return -1.0;
    }
    val
}

fn main() {
    println!("Aggregator logic compiled.");
}
""")

In [6]:
%%writefile aggregator.rs
#[kani::proof]
fn verify_update_bounds() {
    let update: f32 = kani::any();

    // We call our aggregation logic
    let result = safe_aggregate(update);

    // Mathematically prove the result is always within safe bounds
    assert!(result >= -1.0 && result <= 1.0);
}

fn safe_aggregate(val: f32) -> f32 {
    // Handle the NaN bug discovered in the previous run
    if val.is_nan() {
        return 0.0;
    }
    if val > 1.0 {
        return 1.0;
    } else if val < -1.0 {
        return -1.0;
    }
    val
}

fn main() {
    println!("Aggregator logic ready for verification.");
}

Overwriting aggregator.rs


In [7]:
!kani aggregator.rs

Kani Rust Verifier 0.67.0 (standalone)
  --> aggregator.rs:25:4
   |
25 | fn main() {
   |    ^^^^
   |
   = note: `#[warn(dead_code)]` (part of `#[warn(unused)]`) on by default


Checking harness verify_update_bounds...
CBMC 6.8.0 (cbmc-6.8.0)
CBMC version 6.8.0 (cbmc-6.8.0) 64-bit x86_64 linux
Reading GOTO program from file /content/aggregator__RNvCs7Wv7OrDRiPg_10aggregator20verify_update_bounds.out
Generating GOTO Program
Adding CPROVER library (x86_64)
Removal of function pointers and virtual functions
Generic Property Instrumentation
Running with 16 object bits, 48 offset bits (user-specified)
Starting Bounded Model Checking
Runtime Symex: 0.00794452s
size of program expression: 198 steps
slicing removed 87 assignments
Generated 2 VCC(s), 2 remaining after simplification
Runtime Postprocess Equation: 0.000167278s
Passing problem to propositional reduction
converting SSA
Runtime Convert SSA: 0.00230036s
Running propositional reduction
Post-processing
Runtime Post-process: 4.8394e-0

In [8]:
# Save the success output to a file for your repo
with open("verification_report.txt", "w") as f:
    f.write("Kani Rust Verifier - Successful Formal Proof\n")
    f.write("Property: Bound Invariant (-1.0 <= x <= 1.0)\n")
    f.write("Status: VERIFIED\n")
    f.write("Vulnerability Mitigated: IEEE 754 NaN (Not-a-Number) Injection\n")

print("Verification report saved!")

Verification report saved!


In [9]:
import torch

# Simulated updates from 5 nodes
# Node 4 is a malicious actor trying to crash the system with a NaN
# Node 5 is an attacker sending a massive outlier (1000.0)
updates = torch.tensor([0.4, -0.1, 0.8, float('nan'), 1000.0])

def verified_aggregation_sim(tensors):
    # This Python logic mirrors your Verified Rust logic
    # 1. Neutralize NaNs
    clean_tensors = torch.nan_to_num(tensors, nan=0.0)
    # 2. Apply Verified Bounds (-1.0 to 1.0)
    bounded_tensors = torch.clamp(clean_tensors, min=-1.0, max=1.0)
    # 3. Aggregate
    return torch.mean(bounded_tensors)

final_update = verified_aggregation_sim(updates)
print(f"Aggregated Global Update: {final_update.item()}")
print("Result is secure, bounded, and NaN-free!")

Aggregated Global Update: 0.41999998688697815
Result is secure, bounded, and NaN-free!
